In [20]:
import kagglehub
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [21]:
#download dataset from Kaggle
path = kagglehub.dataset_download('new-york-city/nyc-inspections')

csv_file = next(os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv'))

df_raw = pd.read_csv(csv_file, dtype={"PHONE": "string"}, encoding='latin-1', low_memory=False)
df = df_raw.copy()
df['CUISINE DESCRIPTION'] = df['CUISINE DESCRIPTION'].str.replace(
    r'Caf[^\s/]+', 'Café', regex=True
)

print(df.shape)

Using Colab cache for faster access to the 'nyc-inspections' dataset.
(399918, 18)


In [22]:
# Fix data types
for col in ['INSPECTION DATE', 'GRADE DATE', 'RECORD DATE']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['ZIPCODE'] = df['ZIPCODE'].apply(lambda x: str(int(x)).zfill(5) if pd.notna(x) else np.nan)

df['CAMIS'] = df['CAMIS'].astype(str)

In [23]:
# drop exact duplicate rows
df = df.drop_duplicates(keep='first')
print(f'Rows after dedup: {len(df):,}')

Rows after dedup: 399,907


In [24]:
#BORO has 9 rows with the word missing instead of NaN
df['BORO'] = df['BORO'].replace('Missing', np.nan)

In [25]:
# 1135 rows have inspection date = 1900-01-01
sentinel_mask = df['INSPECTION DATE'] == pd.Timestamp('1900-01-01')
df['IS_SENTINEL_DATE'] = sentinel_mask
df.loc[sentinel_mask, 'INSPECTION DATE'] = pd.NaT

In [26]:
#negative scores are impossible in the NYC grading system
neg_mask = df['SCORE'] < 0
df['SCORE_INVALID'] = neg_mask
df.loc[neg_mask, 'SCORE'] = np.nan

# scores > 100 are extreme but possible
df['SCORE_EXTREME'] = df['SCORE'] > 100

In [ ]:
df['HAS_GRADE'] = df['GRADE'].notna()

df['PHONE'] = df['PHONE'].astype("string").str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True)

df['PHONE'] = df['PHONE'].where(df['PHONE'].str.len() == 10, None)

In [28]:
df['VIOLATION_MISMATCH'] = df['VIOLATION CODE'].isna() ^ df['VIOLATION DESCRIPTION'].isna()

In [29]:
df = df.drop(columns=['RECORD DATE'])

In [30]:
#drop rows with no real inspection date
df = df[df['IS_SENTINEL_DATE'] == False]

#drop rows where violation code and description don't match
df = df[df['VIOLATION_MISMATCH'] == False]

In [31]:
# Final checks
print(f'Shape:{df.shape}')
print(f'udplicates:{df.duplicated().sum()}')
print(f'negative scores:{(df["SCORE"] < 0).sum()}')
print(f'dentinel dates:{(df["INSPECTION DATE"] == pd.Timestamp("1900-01-01")).sum()}')
print(f'BORO Missing left:{(df["BORO"] == "Missing").sum()}')

Shape:(398302, 22)
udplicates:0
negative scores:0
dentinel dates:0
BORO Missing left:0


In [ ]:
df.to_csv('nyc_restaurant_inspections_CLEAN.csv', index=False, encoding='utf-8')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>